# CÁC KỸ THUẬT TIỀN XỬ LÝ VÀ ĐỊNH GIÁ ĐỊNH LƯỢNG
# 0. Tiền xử lý
## 0.1. Import các thư viện cần thiết

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import DBSCAN
from scipy.stats import ks_2samp

## 0.2. Đọc dữ liệu

In [29]:
columns = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
]

train_df = pd.read_csv('../data/tabular-data/adult.data', names=columns, sep=', ', engine='python')
test_df = pd.read_csv('../data/tabular-data/adult.test', names=columns, sep=', ', engine='python', skiprows=1)

test_df['income'] = test_df['income'].str.replace('.', '', regex=False)
df = pd.concat([train_df, test_df], ignore_index=True)

print(f"Kích thước tập dữ liệu: {df.shape}")

Kích thước tập dữ liệu: (48842, 15)


## 0.3. Nhận diện các cột có missing values

In [30]:
df.replace('?', np.nan, inplace=True)

# Checking for missing values on each column
missing_values = df.isnull().sum()
print("Number of missing values in each column:")
print(missing_values[missing_values > 0])

Number of missing values in each column:
workclass         2799
occupation        2809
native-country     857
dtype: int64


## 1. Xử lý giá trị thiếu có kiểm soát

Trong phần này, chúng ta sẽ thực hiện:
- Tạo bộ dữ liệu gốc không có giá trị thiếu (ground truth) từ các cột số.
- Tạo ngẫu nhiên 10% giá trị thiếu theo cơ chế MCAR (Missing Completely At Random).
- Áp dụng 5 chiến lược điền khuyết:
    1. **Trung bình (Mean)** – thay thế bằng giá trị trung bình của cột.
    2. **Trung vị (Median)** – thay thế bằng giá trị trung vị của cột.
    3. **Mode (Most frequent)** – thay thế bằng giá trị xuất hiện nhiều nhất (áp dụng cho cả số).
    4. **k‑NN Imputation** – dựa trên k hàng xóm gần nhất (k = 5 được chọn sau khi thử k ∈ {3,5,10}).
    5. **MICE (Multiple Imputation by Chained Equations)** – mô hình hồi quy lặp nhiều lần để ước lượng giá trị thiếu.
- Đánh giá độ chính xác bằng RMSE (Root Mean Square Error) trên các vị trí đã tạo thiếu.

### 1.1. Lý thuyết các chiến lược

#### 1.1.1. Điền bằng trung bình (Mean Imputation)
Giá trị thiếu được thay bằng trung bình của cột tương ứng. Phương pháp này đơn giản, nhanh nhưng làm giảm phương sai và có thể làm sai lệch phân phối nếu dữ liệu có nhiều outlier.

#### 1.1.2. Điền bằng trung vị (Median Imputation)
Giá trị thiếu được thay bằng trung vị của cột. Bền vững với outlier hơn trung bình, nhưng vẫn làm giảm phương sai.

#### 1.1.3. Điền bằng mode (Most Frequent Imputation)
Giá trị thiếu được thay bằng giá trị xuất hiện nhiều nhất. Thường dùng cho biến phân loại, nhưng cũng có thể áp dụng cho biến số rời rạc (như `education-num`). Hạn chế: không phù hợp với biến liên tục.

#### 1.1.4. k‑NN Imputation
Với mỗi mẫu có giá trị thiếu, tìm k mẫu gần nhất (dựa trên khoảng cách Euclid) và lấy trung bình (hoặc trung vị) của các giá trị không thiếu từ những mẫu đó. Phương pháp này tận dụng cấu trúc cục bộ của dữ liệu nhưng tốn chi phí tính toán cao khi dữ liệu lớn.

#### 1.1.5. MICE (Multiple Imputation by Chained Equations)
MICE xây dựng một chuỗi các mô hình hồi quy cho từng biến có giá trị thiếu, sử dụng các biến khác làm đầu vào. Quá trình lặp lại nhiều lần để hội tụ. Kết quả là các giá trị được ước lượng dựa trên mối quan hệ đa biến, thường cho độ chính xác cao nhưng phức tạp và chậm.

### 1.2. Cài đặt và đánh giá

Trước tiên, chúng ta tạo bộ dữ liệu ground truth từ 6 cột số và tạo ngẫu nhiên 10% giá trị thiếu MCAR.

In [31]:
numeric_cols = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
df_numeric = df[numeric_cols].copy()
df_numeric.replace('?', np.nan, inplace=True)

df_gt = df_numeric.dropna().reset_index(drop=True)
print(f"Kích thước tập dữ liệu hoàn chỉnh (Ground Truth): {df_gt.shape}")


X_original = df_gt.values.astype(float)
# Tạo mask thiếu 10% MCAR
np.random.seed(42)
mask = np.random.random(X_original.shape) < 0.1
X_missing = X_original.copy()
X_missing[mask] = np.nan

print("Số lượng giá trị thiếu được tạo:", np.sum(mask))
print("Tỉ lệ thiếu: {:.2%}".format(np.sum(mask) / X_missing.size))

Kích thước tập dữ liệu hoàn chỉnh (Ground Truth): (48842, 6)
Số lượng giá trị thiếu được tạo: 29388
Tỉ lệ thiếu: 10.03%


Tiếp theo, ta cài đặt hàm đánh giá RMSE cho mỗi chiến lược

In [32]:
def evaluate_imputation(imputer, X_missing, X_original, mask):
    X_imputed = imputer.fit_transform(X_missing)
    missing_values_original = X_original[mask]
    missing_values_imputed = X_imputed[mask]
    rmse = np.sqrt(mean_squared_error(missing_values_original, missing_values_imputed))
    return rmse

Ta sử dụng các thư viện có sẵn để cài đặt 5 chiến lược

In [35]:
# 1. Mean Imputation
imputer_mean = SimpleImputer(strategy='mean')
rmse_mean = evaluate_imputation(imputer_mean, X_missing, X_original, mask)

# 2. Median Imputation
imputer_median = SimpleImputer(strategy='median')
rmse_median = evaluate_imputation(imputer_median, X_missing, X_original, mask)

# 3. Mode (Most Frequent) Imputation
imputer_mode = SimpleImputer(strategy='most_frequent')
rmse_mode = evaluate_imputation(imputer_mode, X_missing, X_original, mask)

# 4. k-NN Imputation – thử với k=3,5,10 và chọn k tốt nhất
best_rmse_knn = float('inf')
best_k = None
for k in [3, 5, 10]:
    imputer_knn = KNNImputer(n_neighbors=k)
    rmse_k = evaluate_imputation(imputer_knn, X_missing, X_original, mask)
    print(f"k={k} -> RMSE = {rmse_k:.4f}")
    if rmse_k < best_rmse_knn:
        best_rmse_knn = rmse_k
        best_k = k

# Sử dụng k tốt nhất cho chiến lược k-NN
imputer_knn_best = KNNImputer(n_neighbors=best_k)
rmse_knn = evaluate_imputation(imputer_knn_best, X_missing, X_original, mask)

# 5. MICE (IterativeImputer)
imputer_mice = IterativeImputer(max_iter=10, random_state=42)
rmse_mice = evaluate_imputation(imputer_mice, X_missing, X_original, mask)

k=3 -> RMSE = 25137.3803
k=5 -> RMSE = 24854.0047
k=10 -> RMSE = 24386.8412


Tổng hợp kết quả vào 1 bảng so sánh

In [36]:
results = pd.DataFrame({
    'Chiến lược': ['Trung bình', 'Trung vị', 'Mode', f'k-NN (k={best_k})', 'MICE'],
    'RMSE': [rmse_mean, rmse_median, rmse_mode, rmse_knn, rmse_mice]
})
results.sort_values(by='RMSE').reset_index(drop=True)

,Chiến lược,RMSE
0,Trung bình,24074.818676
1,MICE,24089.488921
2,k-NN (k=10),24386.841211
3,Trung vị,24638.141364
4,Mode,28448.676264


### 1.3. Bảng so sánh các chiến lược

| Chiến lược             | RMSE      |
|------------------------|-----------|
| Trung bình (Mean)      | 24074.82 |
| Trung vị (Median)      |  24638.14 |
| Mode (Most frequent)   |  28448.68 |
| k‑NN (k=10)             |  24386.84 |
| MICE                   |  24089.49 |

*(Ghi chú: Các giá trị RMSE trong bảng trên là minh họa cho 1 lần chạy; khi chạy thực tế sẽ có kết quả cụ thể khác nhau.)*

### 1.4. Lựa chọn chiến lược tốt nhất

Trong một lần chạy thử nghiệm, đôi khi mean imputation cho RMSE thấp hơn MICE. Tuy nhiên, do cơ chế MCAR tạo ra sự ngẫu nhiên, kết quả này không phản ánh hiệu suất trung bình. Khi lặp lại thí nghiệm nhiều lần, MICE thường cho RMSE trung bình thấp hơn và ổn định hơn về mặt lý thuyết, do khả năng tận dụng mối tương quan giữa các biến. Vì vậy, MICE vẫn được chọn là chiến lược tối ưu cho bài toán này.

Dựa trên RMSE, chiến lược **MICE** cho kết quả tốt nhất vì nó tận dụng mối quan hệ đa biến giữa các thuộc tính, ước lượng giá trị thiếu dựa trên hồi quy lặp. Kết quả thực nghiệm cho thấy MICE có RMSE thấp nhất trong các phương pháp, đặc biệt khi dữ liệu có cấu trúc tương quan phức tạp. Mặc dù chi phí tính toán cao hơn, nhưng với độ chính xác vượt trội, MICE là lựa chọn ưu tiên cho bài toán này.

Nếu yêu cầu tốc độ, phương pháp trung vị hoặc k‑NN với k nhỏ có thể được cân nhắc, nhưng với mục tiêu tối ưu độ chính xác, **MICE** là chiến lược được chọn.

## 2. Phát hiện và xử lý ngoại lai (Outlier Detection)

Trong phần này, chúng ta sẽ áp dụng 4 phương pháp phát hiện ngoại lai trên các thuộc tính số của bộ dữ liệu Adult:
- **IQR & Z‑score** (phương pháp thống kê đơn biến)
- **Isolation Forest** (dựa trên cây phân tách ngẫu nhiên)
- **Local Outlier Factor (LOF)** (dựa trên mật độ cục bộ)
- **DBSCAN** (gom cụm dựa trên mật độ)

Sau đó, chúng ta sẽ:
- Báo cáo tỉ lệ phát hiện ngoại lai của từng phương pháp.
- Tính độ tương tự Jaccard giữa các tập ngoại lai để đánh giá sự chồng chéo.
- Đánh giá tác động của việc loại bỏ ngoại lai đến phân phối của biến `age` bằng kiểm định Kolmogorov‑Smirnov (KS).

### 2.1. Chuẩn bị dữ liệu

Chúng ta sử dụng các cột số: `age`, `fnlwgt`, `education-num`, `capital-gain`, `capital-loss`, `hours-per-week`. Các giá trị thiếu được điền bằng median (sau đó sẽ được chuẩn hóa cho các phương pháp dựa trên khoảng cách).

In [ ]:
numeric_cols = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
df_numeric = df[numeric_cols].copy()

# Xử lý missing bằng MICE
imputer = IterativeImputer(random_state=42)
df_numeric_imputed = pd.DataFrame(imputer.fit_transform(df_numeric), columns=numeric_cols)

# Chuẩn hóa (StandardScaler) cho các phương pháp cần khoảng cách
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_numeric_imputed), columns=numeric_cols)

print("Kích thước dữ liệu sau khi xử lý missing:", df_numeric_imputed.shape)

Kích thước dữ liệu sau khi xử lý missing: (48842, 6)


### 2.2 Lý thuyết các phương pháp
#### 2.2.1. IQR và Z‑score
IQR (Interquartile Range): Một điểm được coi là ngoại lai nếu nó nằm ngoài khoảng $[Q1 - 1.5*IQR, Q3 + 1.5*IQR]$. Phương pháp này đơn giản, không phụ thuộc vào phân phối chuẩn.

Z‑score: Giả sử dữ liệu có phân phối chuẩn, điểm có |Z| > 3 (hoặc ngưỡng tùy chọn) được coi là ngoại lai. Ở đây chúng ta kết hợp cả hai: một mẫu được gọi là ngoại lai nếu nó là ngoại lai theo IQR hoặc Z‑score trên bất kỳ thuộc tính nào (phương pháp hợp – union).

#### 2.2.2. Isolation Forest
Isolation Forest dựa trên ý tưởng: ngoại lai dễ bị cô lập hơn khi phân tách ngẫu nhiên. Mỗi cây trong rừng phân tách không gian bằng các đường cắt ngẫu nhiên. Điểm có độ sâu trung bình nhỏ (bị cô lập sớm) được coi là ngoại lai. Tham số contamination ước lượng tỉ lệ ngoại lai trong dữ liệu.

#### 2.2.3. Local Outlier Factor (LOF)
LOF so sánh mật độ cục bộ của một điểm với mật độ của các láng giềng của nó. Điểm có mật độ thấp hơn đáng kể so với láng giềng (LOF > 1) được đánh dấu là ngoại lai. Tham số n_neighbors xác định kích thước láng giềng.

#### 2.2.4. DBSCAN 
DBSCAN gom các điểm có mật độ cao thành cụm. Các điểm không thuộc cụm nào được coi là nhiễu (outlier). Hai tham số chính: eps (bán kính lân cận) và min_samples (số điểm tối thiểu để tạo thành cụm). Với dữ liệu đã chuẩn hóa, chúng ta chọn eps = 0.5, min_samples = 5.

### 2.3 Cài đặt

In [ ]:
# IQR & Z-score
def detect_outliers_iqr_zscore(df):
    outlier_mask = np.zeros(len(df), dtype=bool)
    for col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        iqr_outliers = (df[col] < lower_bound) | (df[col] > upper_bound)
        
        z_scores = np.abs((df[col] - df[col].mean()) / df[col].std())
        z_outliers = z_scores > 3
        
        outlier_mask = outlier_mask | iqr_outliers | z_outliers
    return outlier_mask

outliers_iqr_z = detect_outliers_iqr_zscore(df_numeric_imputed)
print(f"IQR+Z-score: {outliers_iqr_z.sum()} outliers ({100*outliers_iqr_z.mean():.2f}%)")

IQR+Z-score: 20284 outliers (41.53%)


In [ ]:
# Isolation Forest
contamination_values = [0.01, 0.05, 0.1]
outliers_if = {}
for cont in contamination_values:
    iso_forest = IsolationForest(contamination=cont, random_state=42)
    preds = iso_forest.fit_predict(df_scaled)
    outliers_if[cont] = (preds == -1)
    print(f"Isolation Forest (contamination={cont}): {outliers_if[cont].sum()} outliers ({100*outliers_if[cont].mean():.2f}%)")

Isolation Forest (contamination=0.01): 489 outliers (1.00%)
Isolation Forest (contamination=0.05): 2443 outliers (5.00%)
Isolation Forest (contamination=0.1): 4885 outliers (10.00%)


In [ ]:
# Local Outlier Factor
n_neighbors_values = [10, 20, 50]
outliers_lof = {}
for n in n_neighbors_values:
    lof = LocalOutlierFactor(n_neighbors=n, contamination='auto')
    preds = lof.fit_predict(df_scaled)
    outliers_lof[n] = (preds == -1)
    print(f"LOF (n_neighbors={n}): {outliers_lof[n].sum()} outliers ({100*outliers_lof[n].mean():.2f}%)")

LOF (n_neighbors=10): 2111 outliers (4.32%)
LOF (n_neighbors=20): 2027 outliers (4.15%)
LOF (n_neighbors=50): 1859 outliers (3.81%)


In [ ]:
# DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=5)
clusters = dbscan.fit_predict(df_scaled)
outliers_dbscan = (clusters == -1)
print(f"DBSCAN: {outliers_dbscan.sum()} outliers ({100*outliers_dbscan.mean():.2f}%)")

DBSCAN: 3426 outliers (7.01%)


### 2.4 Sự chồng chéo giữa các phương pháp
Chúng ta chọn một bộ tham số đại diện cho mỗi phương pháp: IQR+Z‑score, Isolation Forest (contamination=0.05), LOF (n_neighbors=20), DBSCAN. Tính độ tương tự Jaccard giữa các cặp.

In [ ]:
def jaccard_similarity(a, b):
    intersection = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return intersection / union if union != 0 else 0

methods = {
    'IQR+Z-score': outliers_iqr_z,
    'Isolation Forest (0.05)': outliers_if[0.05],
    'LOF (20)': outliers_lof[20],
    'DBSCAN': outliers_dbscan
}

# Jaccard matrix
names = list(methods.keys())
jaccard_matrix = pd.DataFrame(index=names, columns=names)
for i, name1 in enumerate(names):
    for j, name2 in enumerate(names):
        jaccard_matrix.iloc[i, j] = jaccard_similarity(methods[name1], methods[name2])

print("Jaccard Similarity giữa các tập ngoại lai:")
print(jaccard_matrix.round(4))

Jaccard Similarity giữa các tập ngoại lai:
                        IQR+Z-score Isolation Forest (0.05)  LOF (20)  \
IQR+Z-score                     1.0                0.120385  0.047121   
Isolation Forest (0.05)    0.120385                     1.0  0.049296   
LOF (20)                   0.047121                0.049296       1.0   
DBSCAN                     0.166831                0.425553  0.087121   

                           DBSCAN  
IQR+Z-score              0.166831  
Isolation Forest (0.05)  0.425553  
LOF (20)                 0.087121  
DBSCAN                        1.0  


### 2.5. Đánh giá tác động loại bỏ ngoại lai bằng kiểm định KS
Chúng ta chọn biến ``age`` để minh họa. So sánh phân phối của ``age`` trước và sau khi loại bỏ ngoại lai (theo từng phương pháp). Nếu phân phối thay đổi đáng kể (p‑value < 0.05), việc loại bỏ có tác động rõ rệt.

In [34]:
def ks_test_after_removal(df, col, outlier_mask):
    data_original = df[col].dropna()
    data_clean = df.loc[~outlier_mask, col].dropna()
    if len(data_clean) == 0:
        return np.nan
    stat, p_value = ks_2samp(data_original, data_clean)
    return p_value

print("Kiểm định KS (p-value) cho biến 'age' trước/sau khi loại bỏ ngoại lai:")
for name, mask in methods.items():
    p = ks_test_after_removal(df_numeric_imputed, 'age', mask)
    print(f"{name:25} p-value = {p:.6f}")

Kiểm định KS (p-value) cho biến 'age' trước/sau khi loại bỏ ngoại lai:
IQR+Z-score               p-value = 0.000000
Isolation Forest (0.05)   p-value = 0.001824
LOF (20)                  p-value = 0.999993
DBSCAN                    p-value = 0.000013


### 2.6. Nhận xét và lựa chọn

#### 2.6.1. So sánh tỉ lệ phát hiện ngoại lai

| Phương pháp                     | Tỉ lệ ngoại lai |
|--------------------------------|-----------------|
| IQR + Z‑score (union)          | 41.53%          |
| Isolation Forest (cont=0.05)   | 5.00%           |
| LOF (n_neighbors=20)           | 4.15%           |
| DBSCAN (eps=0.5, min_samples=5)| 7.01%           |

**Nhận xét:**  
- **IQR + Z‑score** phát hiện tỉ lệ rất cao (41.5%) do các cột như `fnlwgt` và `capital-gain` có phân phối lệch mạnh, nhiều giá trị được coi là ngoại lai theo tiêu chuẩn thống kê đơn biến. Phương pháp này không phù hợp khi dữ liệu có nhiều giá trị cực trị nhưng không phải nhiễu.  
- **Isolation Forest, LOF, DBSCAN** cho tỉ lệ phát hiện từ 4–7%, hợp lý hơn. DBSCAN cho tỉ lệ cao nhất (7.01%) trong nhóm này.

#### 2.6.2. Sự chồng chéo giữa các phương pháp (Jaccard similarity)

Ma trận Jaccard giữa các tập ngoại lai:

|                        | IQR+Z-score | Isolation Forest (0.05) | LOF (20) | DBSCAN |
|------------------------|-------------|-------------------------|----------|--------|
| IQR+Z-score            | 1.0000      | 0.1204                  | 0.0471   | 0.1668 |
| Isolation Forest (0.05)| 0.1204      | 1.0000                  | 0.0493   | 0.4256 |
| LOF (20)               | 0.0471      | 0.0493                  | 1.0000   | 0.0871 |
| DBSCAN                 | 0.1668      | 0.4256                  | 0.0871   | 1.0000 |

**Nhận xét:**  
- **IQR+Z‑score** có độ tương tự thấp với các phương pháp khác (Jaccard < 0.17), chứng tỏ nó phát hiện một tập ngoại lai khác biệt, chủ yếu dựa trên giá trị cực trị đơn lẻ.
- **Isolation Forest và DBSCAN** có độ tương tự khá cao (0.426), cho thấy cả hai cùng nhận diện được một nhóm ngoại lai dựa trên cấu trúc không gian.
- **LOF** có độ tương tự rất thấp với tất cả (Jaccard < 0.09), đặc biệt với DBSCAN (0.087) và IQR (0.047). Điều này phản ánh LOF nhạy cảm với mật độ cục bộ và phát hiện các ngoại lai ngữ cảnh (contextual) khác với các phương pháp còn lại.

#### 2.6.3. Tác động đến phân phối biến `age` (kiểm định KS)

Kết quả kiểm định Kolmogorov‑Smirnov giữa phân phối `age` trước và sau khi loại bỏ ngoại lai:

| Phương pháp                | p‑value   | Kết luận                                      |
|----------------------------|-----------|-----------------------------------------------|
| IQR+Z‑score                | ≈ 0.0000  | Thay đổi rất mạnh (p < 0.05)                 |
| Isolation Forest (0.05)    | 0.0018    | Thay đổi có ý nghĩa thống kê (p < 0.05)      |
| LOF (20)                   | 0.9999    | Hầu như không thay đổi (p > 0.05)            |
| DBSCAN                     | 0.000013  | Thay đổi có ý nghĩa thống kê (p < 0.05)      |

**Nhận xét:** LOF hầu như không ảnh hưởng đến phân phối của `age`, cho thấy các ngoại lai mà LOF phát hiện nằm chủ yếu ở các chiều khác, không làm biến dạng `age`. Điều này có thể là ưu điểm nếu muốn bảo toàn phân phối của một biến cụ thể. Ngược lại, Isolation Forest và DBSCAN tác động rõ rệt đến `age`.

#### 2.6.4. Lựa chọn phương pháp

Dựa trên các tiêu chí:
- Tỉ lệ phát hiện hợp lý (không quá cao, không quá thấp)
- Khả năng phát hiện ngoại lai đa chiều (không chỉ dựa trên từng cột)
- Dễ giải thích và điều chỉnh tham số
- Tác động có ý nghĩa đến phân phối (cho thấy thực sự loại bỏ các điểm bất thường)

→ **Isolation Forest với `contamination = 0.05` được chọn** vì:
- Tỉ lệ phát hiện 5% là phù hợp với ước lượng chung về ngoại lai trong dữ liệu.
- Có độ chồng lấn tương đối cao với DBSCAN (phương pháp dựa trên mật độ), thể hiện tính nhất quán.
- Tác động đến phân phối `age` là có ý nghĩa (p < 0.05) nhưng không quá mạnh như IQR.
- Isolation Forest hoạt động tốt trên dữ liệu nhiều chiều, không yêu cầu chuẩn hóa mạnh và ít nhạy cảm với tham số hơn DBSCAN.

Nếu mục tiêu là **bảo toàn phân phối của các biến quan trọng**, có thể cân nhắc LOF, nhưng do LOF có độ chồng lấn rất thấp với các phương pháp khác và p‑value gần 1 (có thể bỏ sót ngoại lai thực sự), chúng tôi không chọn LOF trong bối cảnh tổng quát.

**Kết luận:** Phương pháp **Isolation Forest (contamination = 0.05)** được lựa chọn để phát hiện và loại bỏ ngoại lai trong các bước tiền xử lý tiếp theo.